# Kaggle Playground Series S6E5 — Predict F1 Pit Stops
### Author: Ruide Yin

## Stage 2: Full Pipeline — Feature Engineering & Modeling

Continues from Stage 1 (EDA). 

### 2.0 Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import os
import time
import warnings
import gc
import joblib
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import pickle

warnings.filterwarnings('ignore')
os.makedirs('./outputs', exist_ok=True)

SEED = 12138
N_FOLDS = 5
TARGET = 'PitNextLap'

train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')
print(f"Train: {train.shape}, Test: {test.shape}")
print(f"Positive rate: {train[TARGET].mean():.4f}")

Train: (439140, 16), Test: (188165, 15)
Positive rate: 0.1990


### 2.0.1 Quick Check: RaceProgress Definition

In [2]:
# Verify: is RaceProgress == LapNumber / max(LapNumber) per race-driver-year group?
check = train.copy()
check['MaxLap'] = check.groupby(['Driver', 'Race', 'Year'])['LapNumber'].transform('max')
check['Computed_RP'] = check['LapNumber'] / check['MaxLap']
diff = (check['RaceProgress'] - check['Computed_RP']).abs()
print(f"Max absolute diff:  {diff.max():.6f}")
print(f"Mean absolute diff: {diff.mean():.6f}")
print(f"Exact match rate:   {(diff < 1e-6).mean():.4f}")

# Also check if it could be LapNumber / TotalRaceLaps (not per driver)
check2 = train.copy()
check2['MaxLapRace'] = check2.groupby(['Race', 'Year'])['LapNumber'].transform('max')
check2['Computed_RP2'] = check2['LapNumber'] / check2['MaxLapRace']
diff2 = (check2['RaceProgress'] - check2['Computed_RP2']).abs()
print(f"\nUsing race-level max lap:")
print(f"Max absolute diff:  {diff2.max():.6f}")
print(f"Mean absolute diff: {diff2.mean():.6f}")
print(f"Exact match rate:   {(diff2 < 1e-6).mean():.4f}")
del check, check2; gc.collect()

Max absolute diff:  0.987179
Mean absolute diff: 0.120311
Exact match rate:   0.0142

Using race-level max lap:
Max absolute diff:  0.983051
Mean absolute diff: 0.038551
Exact match rate:   0.1054


NameError: name 'gc' is not defined

## Part 1: Feature Engineering

All features are constructed inside `build_features()` so train and test are processed identically.
Statistics that depend on the target (`PitNextLap`) are computed from train only, then merged to both.

### 2.1 Feature Engineering Functions

In [ ]:
# ── 1.1 Normalized TyreLife approximation (train-only stats) ────────────────
def compute_tyrelife_stats(train_df):
    """Compute pit-lap TyreLife quantiles from train only."""
    pit_rows = train_df[train_df[TARGET] == 1]
    stats = {}
    for group_cols, suffix in [
        (['Compound', 'Race'],    'CR'),
        (['Compound', 'Year'],    'CY'),
        (['Compound', 'Stint'],   'CS'),
        (['Compound', 'Driver'],  'CD'),
    ]:
        g = pit_rows.groupby(group_cols)['TyreLife'].quantile([0.5, 0.75]).unstack()
        g.columns = [f'TL_p50_{suffix}', f'TL_p75_{suffix}']
        g = g.reset_index()
        stats[(tuple(group_cols), suffix)] = g
    return stats

def merge_tyrelife_features(df, stats):
    """Merge pre-computed stats and create normalised TyreLife features."""
    for (group_cols, suffix), stat_df in stats.items():
        df = df.merge(stat_df, on=list(group_cols), how='left')
        p50_col = f'TL_p50_{suffix}'
        p75_col = f'TL_p75_{suffix}'
        df[f'TL_ratio_p50_{suffix}'] = df['TyreLife'] / df[p50_col].replace(0, np.nan)
        df[f'TL_ratio_p75_{suffix}'] = df['TyreLife'] / df[p75_col].replace(0, np.nan)
        df[f'TL_over_p75_{suffix}']  = (df['TyreLife'] > df[p75_col]).astype(np.int8)
        df.drop(columns=[p50_col, p75_col], inplace=True)
    return df


# ── 1.2 Tyre degradation features ──────────────────────────────────────────
def add_degradation_features(df, laptime_q01, laptime_q99, race_year_median):
    """Add degradation-related features."""
    # Average degradation per lap
    df['Deg_per_lap'] = np.where(
        df['TyreLife'] == 0, 0,
        df['Cumulative_Degradation'] / df['TyreLife']
    )
    # Clip LapTime_Delta to 1-99 percentiles
    df['LapTime_Delta_clipped'] = df['LapTime_Delta'].clip(laptime_q01, laptime_q99)
    # LapTime deviation from race+year median
    df = df.merge(race_year_median, on=['Race', 'Year'], how='left')
    df['LapTime_dev'] = df['LapTime (s)'] - df['LapTime_median']
    df.drop(columns=['LapTime_median'], inplace=True)
    return df


# ── 1.3 Race progress features ─────────────────────────────────────────────
def add_progress_features(df):
    df['RemainingProgress'] = 1.0 - df['RaceProgress']
    df['is_last_10pct'] = (df['RaceProgress'] >= 0.90).astype(np.int8)
    df['is_last_5pct']  = (df['RaceProgress'] >= 0.95).astype(np.int8)
    df['is_first_5pct'] = (df['RaceProgress'] <= 0.05).astype(np.int8)
    return df


# ── 1.4 Strategy features ──────────────────────────────────────────────────
COMPOUND_MAP = {'SOFT': 0, 'MEDIUM': 1, 'HARD': 2, 'INTERMEDIATE': 3, 'WET': 4}

def add_strategy_features(df):
    df['is_last_stint'] = (df['Stint'] == df['PitStop'] + 1).astype(np.int8)
    df['Compound_ord'] = df['Compound'].map(COMPOUND_MAP).fillna(2).astype(np.int8)
    return df


# ── 1.5 Target encoding (5-Fold leak-free) + frequency encoding ────────────
def target_encode_column(train_df, test_df, col, target, n_folds=5, seed=42):
    """Return (train_encoded, test_encoded) Series for `col`."""
    global_mean = train_df[target].mean()
    train_enc = pd.Series(np.nan, index=train_df.index, name=f'{col}_te')
    kf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    for trn_idx, val_idx in kf.split(train_df, train_df[target]):
        means = train_df.iloc[trn_idx].groupby(col)[target].mean()
        train_enc.iloc[val_idx] = train_df.iloc[val_idx][col].map(means)
    train_enc.fillna(global_mean, inplace=True)

    full_means = train_df.groupby(col)[target].mean()
    test_enc = test_df[col].map(full_means).fillna(global_mean)
    test_enc.name = f'{col}_te'
    return train_enc, test_enc

def add_encoding_features(train_df, test_df):
    """Add target encoding + frequency encoding for Driver and Race."""
    # Target encoding
    for col in ['Driver', 'Race']:
        tr_te, te_te = target_encode_column(train_df, test_df, col, TARGET)
        train_df[f'{col}_te'] = tr_te.values
        test_df[f'{col}_te']  = te_te.values

    # Frequency encoding (from train counts)
    for col in ['Driver', 'Race']:
        freq = train_df[col].value_counts(normalize=True)
        train_df[f'{col}_freq'] = train_df[col].map(freq).fillna(0)
        test_df[f'{col}_freq']  = test_df[col].map(freq).fillna(0)

    return train_df, test_df


# ── 1.6 Interaction features ───────────────────────────────────────────────
def add_interaction_features(df):
    df['Compound_x_TyreLife']   = df['Compound_ord'] * df['TyreLife']
    df['Compound_x_CumDeg']     = df['Compound_ord'] * df['Cumulative_Degradation']
    df['RaceProgress_x_TyreLife'] = df['RaceProgress'] * df['TyreLife']
    df['Position_x_LTDelta']    = df['Position'] * df['LapTime_Delta_clipped']
    return df


# ── 1.7 Lag features ───────────────────────────────────────────────────────
def add_lag_features(df):
    """Lag features: only when consecutive laps (LapNumber diff == 1)."""
    df = df.sort_values(['Driver', 'Race', 'Year', 'LapNumber']).copy()
    grp = df.groupby(['Driver', 'Race', 'Year'])
    df['LapNum_diff'] = grp['LapNumber'].diff()
    df['LapTime_lag_diff'] = grp['LapTime (s)'].diff()
    df['CumDeg_lag_diff']  = grp['Cumulative_Degradation'].diff()
    # Only keep if consecutive
    mask = df['LapNum_diff'] != 1
    df.loc[mask, 'LapTime_lag_diff'] = np.nan
    df.loc[mask, 'CumDeg_lag_diff']  = np.nan
    df.drop(columns=['LapNum_diff'], inplace=True)
    return df

### 2.2 Build Features

In [ ]:
def build_features(train_df, test_df):
    """Master function: applies all feature engineering to train and test."""
    t0 = time.time()

    # ── Pre-compute stats from train ──
    tl_stats = compute_tyrelife_stats(train_df)
    laptime_q01 = train_df['LapTime_Delta'].quantile(0.01)
    laptime_q99 = train_df['LapTime_Delta'].quantile(0.99)
    race_year_median = (train_df.groupby(['Race', 'Year'])['LapTime (s)']
                        .median().reset_index()
                        .rename(columns={'LapTime (s)': 'LapTime_median'}))

    # ── Apply to both ──
    train_df = merge_tyrelife_features(train_df, tl_stats)
    test_df  = merge_tyrelife_features(test_df,  tl_stats)

    train_df = add_degradation_features(train_df, laptime_q01, laptime_q99, race_year_median)
    test_df  = add_degradation_features(test_df,  laptime_q01, laptime_q99, race_year_median)

    train_df = add_progress_features(train_df)
    test_df  = add_progress_features(test_df)

    train_df = add_strategy_features(train_df)
    test_df  = add_strategy_features(test_df)

    train_df, test_df = add_encoding_features(train_df, test_df)

    train_df = add_interaction_features(train_df)
    test_df  = add_interaction_features(test_df)

    train_df = add_lag_features(train_df)
    test_df  = add_lag_features(test_df)

    print(f"Feature engineering done in {time.time()-t0:.1f}s")
    print(f"Train: {train_df.shape}, Test: {test_df.shape}")
    return train_df, test_df

train, test = build_features(train.copy(), test.copy())

# ── Save engineered datasets ──
train.to_csv('./outputs/train_fe.csv', index=False)
test.to_csv('./outputs/test_fe.csv', index=False)
print(f"Saved: train_fe.csv ({train.shape}), test_fe.csv ({test.shape})")

### 2.3 Define Feature Columns & Prepare Arrays

In [ ]:
drop_cols = ['id', TARGET, 'Driver', 'Compound', 'Race']
FEATURES = [c for c in train.columns if c not in drop_cols]
print(f"Number of features: {len(FEATURES)}")
print(FEATURES)

X = train[FEATURES].values.astype(np.float32)
y = train[TARGET].values.astype(np.float32)
X_test = test[FEATURES].values.astype(np.float32)
test_ids = test['id'].values

# Handle NaN for tree models (they handle natively) — for LR/MLP fill with 0
X_filled = np.nan_to_num(X, nan=0.0)
X_test_filled = np.nan_to_num(X_test, nan=0.0)

# Scale for LR / MLP
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_filled)
X_test_scaled = scaler.transform(X_test_filled)

pos_count = y.sum()
neg_count = len(y) - pos_count
scale_pos = neg_count / pos_count
print(f"\nscale_pos_weight = {scale_pos:.2f}")

## Part 2: Baseline Modeling (StratifiedKFold 5-Fold)

All models use default or reasonable parameters. Class imbalance handled via model parameters.

### 2.4 Baseline Training Harness

In [ ]:
def train_sklearn_model(model, X_tr, y_tr, X_te, name, n_folds=N_FOLDS, use_scaled=False):
    """Generic K-Fold trainer for sklearn-API models. Returns oof_preds, test_preds, auc, elapsed."""
    t0 = time.time()
    oof = np.zeros(len(y_tr))
    test_preds = np.zeros(len(X_te))
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)

    X_use = X_scaled if use_scaled else X_tr
    X_te_use = X_test_scaled if use_scaled else X_te

    for fold, (trn_idx, val_idx) in enumerate(skf.split(X_use, y_tr)):
        clf = model.__class__(**model.get_params())
        clf.fit(X_use[trn_idx], y_tr[trn_idx])
        oof[val_idx] = clf.predict_proba(X_use[val_idx])[:, 1]
        test_preds += clf.predict_proba(X_te_use)[:, 1] / n_folds

    auc = roc_auc_score(y_tr, oof)
    elapsed = time.time() - t0
    print(f"[{name}] OOF AUC = {auc:.6f}  |  Time = {elapsed:.1f}s")
    return oof, test_preds, auc, elapsed


def train_lgb_model(params, X_tr, y_tr, X_te, name, n_folds=N_FOLDS):
    t0 = time.time()
    oof = np.zeros(len(y_tr))
    test_preds = np.zeros(len(X_te))
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)

    for fold, (trn_idx, val_idx) in enumerate(skf.split(X_tr, y_tr)):
        dtrain = lgb.Dataset(X_tr[trn_idx], y_tr[trn_idx])
        dval   = lgb.Dataset(X_tr[val_idx], y_tr[val_idx], reference=dtrain)
        model = lgb.train(params, dtrain, num_boost_round=3000,
                          valid_sets=[dval], callbacks=[
                              lgb.early_stopping(100, verbose=False),
                              lgb.log_evaluation(0)
                          ])
        oof[val_idx] = model.predict(X_tr[val_idx])
        test_preds += model.predict(X_te) / n_folds

    auc = roc_auc_score(y_tr, oof)
    elapsed = time.time() - t0
    print(f"[{name}] OOF AUC = {auc:.6f}  |  Time = {elapsed:.1f}s")
    return oof, test_preds, auc, elapsed


def train_xgb_model(params, X_tr, y_tr, X_te, name, n_folds=N_FOLDS):
    t0 = time.time()
    oof = np.zeros(len(y_tr))
    test_preds = np.zeros(len(X_te))
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)

    for fold, (trn_idx, val_idx) in enumerate(skf.split(X_tr, y_tr)):
        dtrain = xgb.DMatrix(X_tr[trn_idx], y_tr[trn_idx], feature_names=FEATURES)
        dval   = xgb.DMatrix(X_tr[val_idx], y_tr[val_idx], feature_names=FEATURES)
        model = xgb.train(params, dtrain, num_boost_round=3000,
                          evals=[(dval, 'val')],
                          early_stopping_rounds=100, verbose_eval=0)
        oof[val_idx] = model.predict(dval)
        dtest = xgb.DMatrix(X_te, feature_names=FEATURES)
        test_preds += model.predict(dtest) / n_folds

    auc = roc_auc_score(y_tr, oof)
    elapsed = time.time() - t0
    print(f"[{name}] OOF AUC = {auc:.6f}  |  Time = {elapsed:.1f}s")
    return oof, test_preds, auc, elapsed


def train_catboost_model(params, X_tr, y_tr, X_te, name, n_folds=N_FOLDS):
    t0 = time.time()
    oof = np.zeros(len(y_tr))
    test_preds = np.zeros(len(X_te))
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)

    X_tr_filled = np.nan_to_num(X_tr, nan=0.0)
    X_te_filled = np.nan_to_num(X_te, nan=0.0)

    for fold, (trn_idx, val_idx) in enumerate(skf.split(X_tr_filled, y_tr)):
        clf = CatBoostClassifier(**params)
        clf.fit(X_tr_filled[trn_idx], y_tr[trn_idx],
                eval_set=(X_tr_filled[val_idx], y_tr[val_idx]),
                early_stopping_rounds=100, verbose=0)
        oof[val_idx] = clf.predict_proba(X_tr_filled[val_idx])[:, 1]
        test_preds += clf.predict_proba(X_te_filled)[:, 1] / n_folds

    auc = roc_auc_score(y_tr, oof)
    elapsed = time.time() - t0
    print(f"[{name}] OOF AUC = {auc:.6f}  |  Time = {elapsed:.1f}s")
    return oof, test_preds, auc, elapsed

#### 2.4.1 MLP (PyTorch) Training Function

In [ ]:
class MLPClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dims=(256, 128), dropout=0.3):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def train_mlp_model(X_tr, y_tr, X_te, name, hidden_dims=(256, 128),
                    lr=1e-3, dropout=0.3, weight_decay=1e-4, batch_size=4096,
                    epochs=50, n_folds=N_FOLDS):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    t0 = time.time()
    oof = np.zeros(len(y_tr))
    test_preds = np.zeros(len(X_te))
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)

    # Compute class weight
    pos_w = (y_tr == 0).sum() / max((y_tr == 1).sum(), 1)
    pos_weight = torch.tensor([pos_w], dtype=torch.float32).to(device)

    X_te_t = torch.tensor(X_te, dtype=torch.float32).to(device)

    for fold, (trn_idx, val_idx) in enumerate(skf.split(X_tr, y_tr)):
        X_t = torch.tensor(X_tr[trn_idx], dtype=torch.float32)
        y_t = torch.tensor(y_tr[trn_idx], dtype=torch.float32)
        X_v = torch.tensor(X_tr[val_idx], dtype=torch.float32).to(device)

        train_ds = TensorDataset(X_t, y_t)
        train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)

        model = MLPClassifier(X_tr.shape[1], hidden_dims, dropout).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

        best_auc, patience, best_state = 0, 0, None
        for epoch in range(epochs):
            model.train()
            for xb, yb in train_dl:
                xb, yb = xb.to(device), yb.to(device)
                loss = criterion(model(xb).squeeze(), yb)
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            scheduler.step()

            model.eval()
            with torch.no_grad():
                val_logits = model(X_v).squeeze().cpu().numpy()
                val_preds = 1 / (1 + np.exp(-val_logits))
                auc = roc_auc_score(y_tr[val_idx], val_preds)
            if auc > best_auc:
                best_auc = auc; patience = 0
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            else:
                patience += 1
                if patience >= 7:
                    break

        model.load_state_dict(best_state)
        model.eval()
        with torch.no_grad():
            oof[val_idx] = 1 / (1 + np.exp(-model(X_v).squeeze().cpu().numpy()))
            test_preds += (1 / (1 + np.exp(-model(X_te_t).squeeze().cpu().numpy()))) / n_folds

    auc = roc_auc_score(y_tr, oof)
    elapsed = time.time() - t0
    print(f"[{name}] OOF AUC = {auc:.6f}  |  Time = {elapsed:.1f}s")
    return oof, test_preds, auc, elapsed, model

### 2.5 Run All Baselines

In [ ]:
results = {}  # name -> (oof, test_preds, auc, time)

# ── 2.5.1 Logistic Regression ──
lr_model = LogisticRegression(max_iter=1000, class_weight='balanced', C=1.0, random_state=SEED)
oof_lr, tp_lr, auc_lr, t_lr = train_sklearn_model(lr_model, X, y, X_test, 'LR_baseline', use_scaled=True)
results['LR_baseline'] = (oof_lr, tp_lr, auc_lr, t_lr)

# ── 2.5.2 Random Forest ──
rf_model = RandomForestClassifier(n_estimators=500, max_depth=15, min_samples_leaf=20,
                                   class_weight='balanced', random_state=SEED, n_jobs=-1)
oof_rf, tp_rf, auc_rf, t_rf = train_sklearn_model(rf_model, X_filled, y, X_test_filled, 'RF_baseline')
results['RF_baseline'] = (oof_rf, tp_rf, auc_rf, t_rf)

# ── 2.5.3 LightGBM ──
lgb_params_base = {
    'objective': 'binary', 'metric': 'auc', 'verbosity': -1,
    'learning_rate': 0.05, 'num_leaves': 63, 'max_depth': -1,
    'min_child_samples': 50, 'scale_pos_weight': scale_pos,
    'colsample_bytree': 0.8, 'subsample': 0.8, 'subsample_freq': 1,
    'reg_alpha': 0.1, 'reg_lambda': 0.1, 'random_state': SEED,
    'n_jobs': -1,
}
oof_lgb, tp_lgb, auc_lgb, t_lgb = train_lgb_model(lgb_params_base, X, y, X_test, 'LGB_baseline')
results['LGB_baseline'] = (oof_lgb, tp_lgb, auc_lgb, t_lgb)

# ── 2.5.4 XGBoost ──
xgb_params_base = {
    'objective': 'binary:logistic', 'eval_metric': 'auc',
    'learning_rate': 0.05, 'max_depth': 7, 'min_child_weight': 50,
    'scale_pos_weight': scale_pos,
    'colsample_bytree': 0.8, 'subsample': 0.8,
    'reg_alpha': 0.1, 'reg_lambda': 0.1,
    'tree_method': 'hist', 'device': 'cuda',
    'seed': SEED,
}
oof_xgb, tp_xgb, auc_xgb, t_xgb = train_xgb_model(xgb_params_base, X, y, X_test, 'XGB_baseline')
results['XGB_baseline'] = (oof_xgb, tp_xgb, auc_xgb, t_xgb)

# ── 2.5.5 CatBoost ──
cat_params_base = {
    'iterations': 3000, 'learning_rate': 0.05, 'depth': 7,
    'auto_class_weights': 'Balanced',
    'random_seed': SEED, 'task_type': 'GPU', 'verbose': 0,
}
oof_cat, tp_cat, auc_cat, t_cat = train_catboost_model(cat_params_base, X, y, X_test, 'CAT_baseline')
results['CAT_baseline'] = (oof_cat, tp_cat, auc_cat, t_cat)

# ── 2.5.6 MLP ──
oof_mlp, tp_mlp, auc_mlp, t_mlp, mlp_model = train_mlp_model(
    X_scaled.astype(np.float32), y, X_test_scaled.astype(np.float32), 'MLP_baseline')
results['MLP_baseline'] = (oof_mlp, tp_mlp, auc_mlp, t_mlp)

In [ ]:
# Print baseline summary
print("\n" + "="*60)
print("BASELINE RESULTS")
print("="*60)
baseline_df = pd.DataFrame({
    'Model': list(results.keys()),
    'OOF_AUC': [v[2] for v in results.values()],
    'Time_s': [v[3] for v in results.values()],
}).sort_values('OOF_AUC', ascending=False).reset_index(drop=True)
print(baseline_df.to_string(index=False))

In [ ]:
# ── Save everything for Notebook 2 ──
np.save('./outputs/X.npy', X)
np.save('./outputs/y.npy', y)
np.save('./outputs/X_test.npy', X_test)
np.save('./outputs/X_scaled.npy', X_scaled)
np.save('./outputs/X_test_scaled.npy', X_test_scaled)
np.save('./outputs/X_filled.npy', X_filled)
np.save('./outputs/X_test_filled.npy', X_test_filled)
np.save('./outputs/test_ids.npy', test_ids)
pickle.dump(results, open('./outputs/baseline_results.pkl', 'wb'))
pickle.dump(FEATURES, open('./outputs/features_list.pkl', 'wb'))
pickle.dump(float(scale_pos), open('./outputs/scale_pos.pkl', 'wb'))
joblib.dump(scaler, './outputs/scaler.pkl')
print("Notebook 2 done. All saved to ./outputs/")